In [1]:
import pandas as pd
import os

In [2]:
# Importando o caminho onde estão os arquivos da silver:
caminho_base = r"C:\Users\caldo\OneDrive\Desktop\Projetos_GIT\project_01_brazilian_ecommerce\data\2_silver"

# Aqui faço uma simples correção das barras, troco a barra '\' pela '/':
caminho = caminho_base.replace('\\','/') + '/'
print('Caminho base para os arquivos:')
print(caminho)

print('-----' * 10)

# Aqui eu listo os arquivos parquet que estão presentes dentro da pasta silver:
lista = os.listdir(caminho)
print(lista)

Caminho base para os arquivos:
C:/Users/caldo/OneDrive/Desktop/Projetos_GIT/project_01_brazilian_ecommerce/data/2_silver/
--------------------------------------------------
['customers_silver_dataset.parquet', 'geolocalization_silver_dataset.parquet', 'itemns_orders_silver_dataset.parquet', 'orders_silver_dataset.parquet', 'payment_order_silver_dataset.parquet', 'products_silver_dataset.parquet', 'sellers_silver_dataset.parquet']


In [3]:
# Criando a lista com os caminhos completos dos arquivos do silver salvos na pasta 2_silver:
caminhos = []
for arquivo in lista:
    caminho_completo = caminho + arquivo
    caminhos.append(caminho_completo)

for arquivo in caminhos:
    print(arquivo)

C:/Users/caldo/OneDrive/Desktop/Projetos_GIT/project_01_brazilian_ecommerce/data/2_silver/customers_silver_dataset.parquet
C:/Users/caldo/OneDrive/Desktop/Projetos_GIT/project_01_brazilian_ecommerce/data/2_silver/geolocalization_silver_dataset.parquet
C:/Users/caldo/OneDrive/Desktop/Projetos_GIT/project_01_brazilian_ecommerce/data/2_silver/itemns_orders_silver_dataset.parquet
C:/Users/caldo/OneDrive/Desktop/Projetos_GIT/project_01_brazilian_ecommerce/data/2_silver/orders_silver_dataset.parquet
C:/Users/caldo/OneDrive/Desktop/Projetos_GIT/project_01_brazilian_ecommerce/data/2_silver/payment_order_silver_dataset.parquet
C:/Users/caldo/OneDrive/Desktop/Projetos_GIT/project_01_brazilian_ecommerce/data/2_silver/products_silver_dataset.parquet
C:/Users/caldo/OneDrive/Desktop/Projetos_GIT/project_01_brazilian_ecommerce/data/2_silver/sellers_silver_dataset.parquet


In [4]:
# Criando os DataFrames com os caminhos dos arquivos que configuramos:
df_customer = pd.read_parquet(caminhos[0]).drop(columns=['data_ingestao'], errors='ignore')
df_geo = pd.read_parquet(caminhos[1]).drop(columns=['data_ingestao'], errors='ignore')
df_itens_order = pd.read_parquet(caminhos[2]).drop(columns=['data_ingestao'], errors='ignore')
df_order = pd.read_parquet(caminhos[3]).drop(columns=['data_ingestao'], errors='ignore')
df_payment = pd.read_parquet(caminhos[4]).drop(columns=['data_ingestao'], errors='ignore')
df_products = pd.read_parquet(caminhos[5]).drop(columns=['data_ingestao'], errors='ignore')
df_sellers = pd.read_parquet(caminhos[6]).drop(columns=['data_ingestao'], errors='ignore')

## Estruturação da camada GOLD

In [5]:
# Agregação de pagamentos:
    # Na nossa tabela de pagamentos, os IDs se repetem para indicar o pagamento de cada parcela do produto.
    # A ideia aqui é AGREGAR todos esses pagamentos pelo order_id:
# Exemplo de pagamento parcelado:
# df_payment[df_payment['order_id'] == '4069c489933782af79afcd3a0e4d693c']

df_payment_agg = (
    df_payment
    .groupby('order_id', as_index=False)
    .agg({
        'payment_value': 'sum',
        'payment_installments': 'max'
    })
)
df_payment_agg.head()

,order_id,payment_value,payment_installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,3
2,000229ec398224ef6ca0657da4fc703e,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3


In [6]:
# Aqui definimos o tipo de pagamento predominante por cada pedido:

df_payment_type = (
    df_payment
    .groupby('order_id')['payment_type']
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
    .reset_index()
)

df_payment_type.head()

,order_id,payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,credit_card
1,00018f77f2f0320c557190d7a144bdd3,credit_card
2,000229ec398224ef6ca0657da4fc703e,credit_card
3,00024acbcdf0a6daa1e931b038114c75,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,credit_card


In [7]:
# Agregamos os dois DataFrames em um só, consolidando a informação:

df_payment_final = df_payment_agg.merge(df_payment_type, on='order_id', how='left')
df_payment_final.head()

,order_id,payment_value,payment_installments,payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,credit_card


In [8]:
# Montando a gold_sales(OBT):

df_gold_sales = (
    df_itens_order
    .merge(df_order[['order_id','customer_id','order_status','data_compra','tempo_entrega','atrasou_entrega']], 
           on='order_id', how='left')
    .merge(df_customer[['customer_id','customer_city','customer_state']],
          on='customer_id', how='left')
    .merge(df_payment_final[['order_id','payment_installments','payment_type']],
          on='order_id', how='left')
    .merge(df_products[['product_id','product_category_name']],
          on='product_id', how='left')
    .merge(df_sellers[['seller_id','seller_city','seller_state']],
          on='seller_id', how='left')
)

print(df_gold_sales.shape)
df_gold_sales.head()

(112650, 20)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,total_value,customer_id,order_status,data_compra,tempo_entrega,atrasou_entrega,customer_city,customer_state,payment_installments,payment_type,product_category_name,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13,7.0,0.0,campos dos goytacazes,RJ,2.0,credit_card,cool_stuff,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26,16.0,0.0,santa fe do sul,SP,3.0,credit_card,pet_shop,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14,7.0,0.0,para de minas,MG,5.0,credit_card,moveis_decoracao,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08,6.0,0.0,atibaia,SP,2.0,credit_card,perfumaria,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04,25.0,0.0,varzea paulista,SP,3.0,credit_card,ferramentas_jardim,loanda,PR


In [9]:
# Conferindo o nome das colunas:
df_gold_sales.columns.tolist()

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'total_value',
 'customer_id',
 'order_status',
 'data_compra',
 'tempo_entrega',
 'atrasou_entrega',
 'customer_city',
 'customer_state',
 'payment_installments',
 'payment_type',
 'product_category_name',
 'seller_city',
 'seller_state']

In [10]:
# Conferindo valores nulos nas principais colunas:
df_gold_sales[['order_id', 'product_id', 'seller_id', 'customer_id']].isnull().sum()

order_id       0
product_id     0
seller_id      0
customer_id    0
dtype: int64

In [11]:
# Conferindo valores nulos em outras colunas importantes:
df_gold_sales[['product_category_name', 'customer_state', 'seller_state', 'payment_type']].isnull().sum()

product_category_name    1603
customer_state              0
seller_state                0
payment_type                3
dtype: int64

In [12]:
# Conferência final dos schemas:
df_gold_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   order_id               112650 non-null  string        
 1   order_item_id          112650 non-null  string        
 2   product_id             112650 non-null  string        
 3   seller_id              112650 non-null  string        
 4   shipping_limit_date    112650 non-null  datetime64[ns]
 5   price                  112650 non-null  float64       
 6   freight_value          112650 non-null  float64       
 7   total_value            112650 non-null  float64       
 8   customer_id            112650 non-null  string        
 9   order_status           112650 non-null  string        
 10  data_compra            112650 non-null  datetime64[ns]
 11  tempo_entrega          110196 non-null  float64       
 12  atrasou_entrega        110196 non-null  floa

In [15]:
# Copiando caminho base do computador e ajustando:
caminho_destino_gold_raw = r'C:\Users\caldo\OneDrive\Desktop\Projetos_GIT\project_01_brazilian_ecommerce\data\3_gold'
caminho_destino_gold = caminho_destino_gold_raw.replace('\\','/') + '/'
caminho_destino_gold

'C:/Users/caldo/OneDrive/Desktop/Projetos_GIT/project_01_brazilian_ecommerce/data/3_gold/'

In [16]:
# Salvando arquivo na camada gold:
df_gold_sales.to_parquet(caminho_destino_gold + 'obt_gold_brazillian_ecommerce.parquet', index=False)